# Lab 5: Workflows com LLM (10 min)

## Objetivos
- Encadear múltiplas chamadas ao modelo
- Criar pipelines de processamento de texto
- Usar output de um passo como input do seguinte

## Conceitos

### O que é um Workflow LLM?
Um pipeline onde cada passo usa o modelo para uma tarefa específica:

```
Texto original
    │
    ▼
[Passo 1: Análise]     → Entender o conteúdo
    │
    ▼
[Passo 2: Transformação] → Processar/transformar
    │
    ▼
[Passo 3: Output]       → Gerar resultado final
```

Cada passo tem o seu próprio **system prompt** especializado.

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

def executar_passo(system_prompt: str, user_input: str, step_name: str) -> str:
    """Executa um passo do workflow."""
    print(f"\n⚙️ {step_name}...")
    response = client.complete(
        model=model,
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_input),
        ],
        max_tokens=500,
        temperature=0.3,
    )
    resultado = response.choices[0].message.content
    print(f"   Tokens: {response.usage.total_tokens}")
    return resultado

print("✅ Setup completo")

## 🖥️ Exercício 5.1: Pipeline de Processamento de Texto

Vamos criar um pipeline que:
1. **Analisa** um texto técnico
2. **Simplifica** para audiência não-técnica
3. **Traduz** para inglês

In [ ]:
# Exercício 5.1: Pipeline de texto
texto_original = """
O Azure AI Foundry utiliza uma arquitetura hub-spoke onde o hub centraliza 
a gestão de recursos partilhados como Azure AI Services, Key Vault e Storage Account,
enquanto os projetos (spokes) herdam esses recursos e adicionam configurações específicas.
Os modelos são deployados via endpoints serverless (MaaS) ou managed compute,
e o acesso é controlado por RBAC com managed identities para autenticação service-to-service.
"""

print(f"📄 Texto original ({len(texto_original.split())} palavras)")

# Passo 1: Análise
analise = executar_passo(
    "Analisa o texto técnico e identifica: 1) Tema principal, 2) Conceitos-chave, 3) Nível de complexidade (1-5). Responde em formato estruturado.",
    texto_original,
    "Passo 1: Análise",
)
print(f"\n{analise}")

# Passo 2: Simplificação
simplificado = executar_passo(
    "Reescreve o seguinte texto técnico para que qualquer pessoa sem conhecimentos de cloud computing consiga entender. Usa analogias do dia-a-dia. Máximo 3 frases.",
    texto_original,
    "Passo 2: Simplificação",
)
print(f"\n{simplificado}")

# Passo 3: Tradução
traduzido = executar_passo(
    "Traduz o seguinte texto para inglês. Mantém o tom simples e acessível.",
    simplificado,
    "Passo 3: Tradução",
)
print(f"\n{traduzido}")

print("\n" + "=" * 60)
print("✅ Pipeline concluído!")

## 🖥️ Exercício 5.2: Pipeline de Análise de Feedback

Pipeline mais complexo com análise de sentimento e categorização.

In [ ]:
# Exercício 5.2: Pipeline de análise de feedback
import json

feedbacks = [
    "O portal do Foundry é muito intuitivo, adorei a experiência de deploy dos modelos!",
    "Demorei 3 horas a configurar a autenticação. A documentação é confusa.",
    "O preço é razoável para empresas, mas startups precisam de tier gratuito maior.",
    "O code interpreter nos agentes é fantástico para análise de dados.",
]

# Passo 1: Classificar sentimento de cada feedback
classificacao = executar_passo(
    """Classifica cada feedback com:
- sentimento: positivo/negativo/neutro
- categoria: UX, documentação, preço, funcionalidade
- prioridade: alta/média/baixa
Responde em JSON válido como array de objetos.""",
    "\n".join([f"{i+1}. {f}" for i, f in enumerate(feedbacks)]),
    "Passo 1: Classificação",
)
print(f"\n{classificacao}")

# Passo 2: Gerar relatório executivo
relatorio = executar_passo(
    "Com base na classificação dos feedbacks, gera um relatório executivo de 5 linhas com: resumo geral, pontos fortes, pontos a melhorar, e uma recomendação prioritária.",
    classificacao,
    "Passo 2: Relatório Executivo",
)
print(f"\n{relatorio}")

## ✅ Resumo

Neste lab aprendeste:
- A encadear chamadas ao modelo em **pipelines**
- Que cada passo pode ter um **system prompt especializado**
- A usar output de um passo como **input do seguinte**
- Padrões: análise → transformação → output

**Próximo:** [Lab 5b - Workflow de Agentes](lab05b-agent-workflows.ipynb)